In [1]:
# Core libraries
import pandas as pd
import numpy as np
import re
from datetime import datetime

# The star of the show
from google_play_scraper import app, reviews, Sort

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [2]:
# Step 1: Get app metadata (rating, installs, description...) for CBE, BOA, Dashen banks

# Map clean display names to their respective Play Store App IDs

banks = {
    "Commercial Bank of Ethiopia": "com.combanketh.mobilebanking",
    "Bank of Abyssinia": "com.boa.boaMobileBanking",
    "Dashen Bank": "com.dashen.dashensuperapp"
}

# Iteratively fetch and display app metadata
for bank_name, app_id in banks.items():
    try:
    # Fetch data directly inside the loop using the current app_id
        app_info = app(app_id, lang='en', country='et')     # Language: English, Country: Ethiopia
    
        print("=" * 50)
        print(f"{bank_name} App Info")
        print("=" * 50)
        print(f"App Title    : {app_info['title']}")
        print(f"Current Score: {app_info['score']}")
        print(f"Total Ratings: {app_info['ratings']:,}")
        print(f"Total Reviews: {app_info['reviews']:,}")
        print(f"Installs     : {app_info['installs']}\n")

    except Exception as e:
        # If an ID fails (like a 404), catch it here and keep going
        print("=" * 50)
        print(f"⚠️ Error loading {bank_name}")
        print("=" * 50)
        print(f"Could not retrieve App ID: '{app_id}'")
        print(f"Details: {e}\n")

Commercial Bank of Ethiopia App Info
App Title    : Commercial Bank of Ethiopia
Current Score: 4.289424
Total Ratings: 48,384
Total Reviews: 9,316
Installs     : 5,000,000+

Bank of Abyssinia App Info
App Title    : BoA Mobile
Current Score: 4.3876925
Total Ratings: 9,234
Total Reviews: 1,461
Installs     : 1,000,000+

Dashen Bank App Info
App Title    : Dashen Bank
Current Score: 4.2579503
Total Ratings: 5,644
Total Reviews: 1,023
Installs     : 1,000,000+



In [7]:
# Step 2: Scrape reviews

# Dictionary to map clean display names to their App IDs
banks = {
    "Commercial Bank of Ethiopia": "com.combanketh.mobilebanking",
    "Bank of Abyssinia": "com.boa.boaMobileBanking",
    "Dashen Bank": "com.dashen.dashensuperapp"
}

# Master dictionary to store the actual raw reviews lists for each bank
all_bank_reviews = {}

# 1. Iterate through each bank to scrape reviews
for bank_name, app_id in banks.items():  
    try:
        # Scrape reviews
        result, continuation_token = reviews(
            app_id,
            lang='en',
            country='et',
            sort=Sort.NEWEST,       # Most recent first
            count=500,              # Target count
            filter_score_with=None  # All star ratings
        )
        
        # Store the list of reviews using the bank's name as the key
        all_bank_reviews[bank_name] = result
        print(f"✅ Success: Collected {len(result)} raw reviews for {bank_name}\n")
        
    except Exception as e:
        # **Safety Net:** If one bank fails (e.g., a temporary network glitch or an ID change), it catches errors and keep the loop running
        print(f"❌ Error scraping {bank_name} due to error: {e}\n")

# --- FINAL VALIDATION CHECK ---
print("=" * 50)
print("FINAL COLLECTION CHECK SUMMARY")
print("=" * 50)
for bank, status in all_bank_reviews.items():
    print(f"{bank:<30} : {len(status)} Raw Reviews")
print("=" * 50)

# 2. Inspect a single raw review from each bank
print("=" * 60)
print("INSPECTING A SAMPLE REVIEW FOR EACH BANK")
print("=" * 60)

# Iterate through all banks in the collected data
for bank_name, reviews_list in all_bank_reviews.items():
    print(f"\nTarget Bank: {bank_name}")
    print("-" * 50)
    
    if reviews_list and len(reviews_list) > 0:
        # Grab the very first review dictionary from the current bank's list
        sample_review = reviews_list[0]
        
        # Display all the available keys safely
        print(f"Keys available in this review: {list(sample_review.keys())}\n")
        print("First raw review data details:")
        
        for key, value in sample_review.items():
            print(f" {key:<20}: {value}")
            
    else:
        print(f"No sample data available for {bank_name}. Check your collection step.")
    print("-" * 50)

✅ Success: Collected 500 raw reviews for Commercial Bank of Ethiopia

✅ Success: Collected 500 raw reviews for Bank of Abyssinia

✅ Success: Collected 500 raw reviews for Dashen Bank

FINAL COLLECTION CHECK SUMMARY
Commercial Bank of Ethiopia    : 500 Raw Reviews
Bank of Abyssinia              : 500 Raw Reviews
Dashen Bank                    : 500 Raw Reviews
INSPECTING A SAMPLE REVIEW FOR EACH BANK

Target Bank: Commercial Bank of Ethiopia
--------------------------------------------------
Keys available in this review: ['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

First raw review data details:
 reviewId            : ba0c5d66-8085-4bff-908b-f553c7b14ff5
 userName            : Yalew Mamed
 userImage           : https://play-lh.googleusercontent.com/a/ACg8ocIJXtC-Q6pj9HbzXLNKgIBuWYQD_rm08H-GXwurQhvLOC88Yg=mo
 content             : It's not allowing me to transfer money.
 score        

In [14]:
# Step 3: Extract only the columns needed from all banks
combined_raw_data = []

for bank_name, reviews_list in all_bank_reviews.items():
    print(f"Processing and extracting columns for: {bank_name}...")
    
    for r in reviews_list:
        combined_raw_data.append({
            'review_id': r.get('reviewId', ''),
            'review'   : r.get('content', ''),
            'rating'   : r.get('score', None),
            'date'     : r.get('at', None),
            'bank'     : bank_name,          # Dynamically tags the correct bank name
            'source'   : 'Google Play'
        })

# Build a single master DataFrame
df_raw = pd.DataFrame(combined_raw_data)

print("\n" + "=" * 50)
print(f"Extraction Complete!")
print(f"Final Combined DataFrame Shape: {df_raw.shape}")
print("=" * 50)

# Display a breakdown of reviews collected per bank
print("\nReviews per bank in DataFrame:")
print(df_raw['bank'].value_counts())

# Randomly select and display 5 reviews in a table format
df_raw.sample(5)

Processing and extracting columns for: Commercial Bank of Ethiopia...
Processing and extracting columns for: Bank of Abyssinia...
Processing and extracting columns for: Dashen Bank...

Extraction Complete!
Final Combined DataFrame Shape: (1500, 6)

Reviews per bank in DataFrame:
bank
Commercial Bank of Ethiopia    500
Bank of Abyssinia              500
Dashen Bank                    500
Name: count, dtype: int64


,review_id,review,rating,date,bank,source
576,497be43a-22d6-41de-b1b8-25b6a9828edb,its the the bank of bank,5,2026-03-25 10:33:59,Bank of Abyssinia,Google Play
1393,befc23dd-22be-4039-ab16-311ebf096360,It takes gazillion years to open 😶,3,2025-10-02 10:54:44,Dashen Bank,Google Play
27,f9d96cc8-f98d-4cd4-9d0c-7ce9af07256d,posetive,5,2026-05-10 07:13:20,Commercial Bank of Ethiopia,Google Play
668,e1556dbc-4ca8-4d4f-a9e5-f2bf4a8559a5,"balance sync is working after 24 hours, freque...",2,2026-02-23 08:12:09,Bank of Abyssinia,Google Play
431,c83ece70-b75a-4917-99d7-29adfe39cad1,proud of CBE. CBE is my everyday day choice an...,5,2026-03-15 15:55:23,Commercial Bank of Ethiopia,Google Play
